# Example notebook computing forecasts and losses using random/trained weights

The prerequisites of the notebook are:

1. The dataset and normalization artifacts.
2. The global ocean mesh.
3. Trained model

All settings are serialized and saved on disk for later reuse.

In [ ]:
import pathlib
import platform

import cartopy.crs as ccrs
import grain.python as grain
import haiku as hk
import jax
import numpy as np
import orbax.checkpoint as ocp
import panel as pn
import treescope
import xarray as xr

from graphcast import cli_utils, training_utils as trn_utils, xarray_jax
from graphcast.dataloader import ARCODataSource
from graphcast.mesh_graph import faces_to_edges
from graphcast.model import TaskConfig

pn.extension()

In [ ]:
# The notebook is thought to be executed both on local hardware and on Leonardo. When running on laptop it uses a coarser grid and mesh.

if platform.node().endswith('leonardo.local'):
    code_path = pathlib.Path("/leonardo_scratch/fast/OGS23_PRACE_IT_0/scampane/xcast")
    data_path = pathlib.Path("/leonardo_scratch/large/userexternal/scampane/xcast/")
    train_path = pathlib.Path("/leonardo_scratch/fast/OGS23_PRACE_IT_0/scampane/xcast/data/training_checkpoints/experiment_190326")
    resolution = "0p25"
    batch_size = 1
else:
    code_path = pathlib.Path("../")
    data_path = pathlib.Path("../")
    train_path = None
    resolution = "1"
    batch_size = 1
    jax.config.update('jax_num_cpu_devices', 4)

In [ ]:
def get_dashboard(dataset, title="", width=600, height=480, absolute_scale=False, **kwargs):
    """ Display a dashboard showing data with possibly all of (batch, time, level, lat, lon) dimensions."""

    variable_selector = pn.widgets.Select(
        name='Variable',
        options=list(dataset.data_vars))

    if 'level' in dataset.coords:
        level_options = {val: idx for idx, val in enumerate(dataset['level'].to_numpy())}
    else:
        level_options = []

    level_selector = pn.widgets.Select(
        name='Level',
        options=level_options)

    if 'time' in dataset.coords:
        time_options = {val: idx for idx, val in enumerate(dataset['time'].dt.days.to_numpy())}
    else:
        time_options = []
        
    time_selector = pn.widgets.DiscreteSlider(
        name='Day',
        options=time_options)
  
    batch_selector = pn.widgets.Select(
        name='Batch',
        options=[] if 'batch' not in dataset.dims else dataset['batch'].to_numpy().tolist())
    
    @pn.depends(variable_selector.param.value, level_selector.param.value, batch_selector.param.value, time_selector.param.value)
    def display(selected_variable, selected_level, selected_batch, selected_time):
        try:
            da = dataset[selected_variable]
            if absolute_scale:
                clim = (da.min(), da.max())
            else:
                clim=None
            if 'level' in da.dims:
                da = da.isel(level=selected_level)
                da = da.drop_vars('level')
            if 'time' in da.dims:
                da = da.isel(time=selected_time)
                da = da.drop_vars('time')
            if 'batch' in da.dims:
                da = da.isel(batch=selected_batch)
            return da.hvplot.image("lon", "lat", clim=clim, width=width, height=height, **kwargs)
        except Exception as e:
            # Return an informative message if an error occurs during plotting
            return pn.pane.Markdown(f"### Error generating plot for var={selected_variable}, level={selected_level}, batch={selected_batch}, time={selected_time}: {e}")

    dashboard = pn.Row(
        pn.Column(
            title,
            variable_selector,
            level_selector,
            batch_selector,
            time_selector),
        display)
    
    return dashboard

In [ ]:
config_path = code_path / "configs/training/launch_full.toml"

configs = cli_utils.Configs.read(config_path)

In [ ]:
mesh_data = trn_utils.get_mesh(data_path, configs)

In [ ]:
def get_max_edge_distance(mesh):
  senders, receivers = faces_to_edges(mesh.faces)
  edge_distances = np.linalg.norm(mesh.vertices[senders] - mesh.vertices[receivers], axis=-1)
  return edge_distances.max()

query_radius = configs.get("model.radius_query_fraction_edge_length", required=True) * get_max_edge_distance(mesh_data.mesh_graph)
print(f"{query_radius=}")

In [ ]:
mask = trn_utils.get_mask(data_path, configs)
grid_lat = mask['lat'].to_numpy()
grid_lon = mask['lon'].to_numpy()
grid_mask = mask.transpose('lat', 'lon').to_numpy()

mean_by_level, stddev_by_level, diffs_stddev_by_level = trn_utils.get_artifacts(data_path, configs)

In [ ]:
get_dashboard(diffs_stddev_by_level, title='# Diffs std by level', projection=ccrs.Robinson())

In [ ]:
dataset_path = data_path / f"data/dataset/dataset_tres-1d_res-{resolution}_levels-10_arco"
from_date = "2021-01-01"

task_config = TaskConfig(
    input_variables=configs.get('task.input_variables', required=True),
    target_variables=configs.get('task.target_variables', required=True),
    forcing_variables=configs.get('task.forcing_variables', required=True),
    levels=configs.get('task.levels', required=True),
    input_duration=configs.get('task.input_duration', required=True))

datasource = ARCODataSource(dataset_path,
                            task=task_config,
                            target_lead_times=configs.get('dataset.target_lead_times', required=True),
                            from_date=from_date,
                            to_date=None)

dataset = grain.MapDataset.source(datasource)

inputs, targets, forcings = dataset[0]

In [ ]:
get_dashboard(inputs, title="# Inputs data from ARCO-OCEAN", projection=ccrs.Robinson())

The location and scale of the variables without `time` dimension (or computed analytically) have been computed as follows.

| Variable name      | Normalization |
|--------------------|---------------|
| `deptho`           | [0, 1]        |
| `waverys_deptho`   | [0, 1]        |
| `tisr`             | [0, 1]        |
| `z`                | [0, 1]        |
| `uparea`           | [0, 1]        |
| `glorys_mask`      | none          |
| `glofas_mask`      | none          |
| `lsm`              | none          |
| `year_progress_sin`| none          |
| `year_progress_cos`| none          |

In [ ]:
# TODO: find out how tisr and progress variables are normalized in original GraphCast code

In [ ]:
@hk.without_apply_rng
@hk.transform
def loss_and_predictions(data, static_data):
    inputs, targets, forcings = data
    mean_by_level, stddev_by_level, diffs_stddev_by_level, mask_da = static_data
    predictor = trn_utils.get_predictor(configs=configs,
                                        mesh_data=mesh_data,
                                        grid_lat=grid_lat,
                                        grid_lon=grid_lon,
                                        grid_mask=grid_mask,
                                        mean_by_level=mean_by_level,
                                        stddev_by_level=stddev_by_level,
                                        diffs_stddev_by_level=diffs_stddev_by_level,
                                        mask_da=mask_da)
    (loss, diagnostics), predictions = predictor.loss_and_predictions(inputs=inputs, targets=targets, forcings=forcings)
    loss, diagnostics = xarray_jax.unwrap_data(loss, require_jax=True), xarray_jax.jax_vars(diagnostics)
    return (loss, diagnostics), predictions

In [ ]:
static_data = xarray_jax.wrap_data((mean_by_level, stddev_by_level, diffs_stddev_by_level, mask), to_jax=True, np_contiguous=False)
data = xarray_jax.device_put((inputs, targets, forcings))
key = jax.random.key(configs.get('seed', required=True))

In [ ]:
initial_params = loss_and_predictions.init(key, data=data, static_data=static_data)

In [ ]:
with treescope.active_autovisualizer.set_scoped(treescope.ArrayAutovisualizer()):
  treescope.display(initial_params)

In [ ]:
loss_and_predictions_jit = jax.jit(loss_and_predictions.apply)

In [ ]:
(initial_loss, initial_diagnostics), random_forecast = loss_and_predictions_jit(params=initial_params, data=data, static_data=static_data)
print(f"{initial_loss=}")

In [ ]:
def to_np(ds):
    return xr.Dataset(data_vars={name: (data.dims, np.asarray(data.data.jax_array)) 
                                 for name, data in ds.data_vars.items()}, 
                      coords=ds.coords, 
                      attrs=ds.attrs)

def get_errors(fc, qmin=0.1, qmax=0.9):
    errs = to_np(fc - targets)
    errs = errs.clip(min=errs.quantile(qmin), max=errs.quantile(qmax))
    return errs

In [ ]:
get_dashboard(get_errors(random_forecast), title="# Random forecast error", projection=ccrs.Robinson())

In [ ]:
ckpt_mngr = trn_utils.get_checkpoint_manager(train_path, configs)
best_step = ckpt_mngr.best_step()
print(f"{best_step=}")
# TODO: check restore and save args options related to sharding and layout
# FIXME: Orbax messes up with global mesh, see comment in get_checkpoint_manager. Check if new versions of Orbax fix the issue.
restored = ckpt_mngr.restore(
    step=best_step,
    args=ocp.args.Composite(params=ocp.args.StandardRestore(initial_params)))
params = jax.device_put(restored.params)

In [ ]:
with treescope.active_autovisualizer.set_scoped(treescope.ArrayAutovisualizer()):
  treescope.display(params)

In [ ]:
(loss, diagnostics), forecast = loss_and_predictions_jit(params=params, data=data, static_data=static_data)
print(f"{loss=}, {diagnostics=}")

In [ ]:
get_dashboard(get_errors(forecast), title="# Forecast error", projection=ccrs.Robinson())

In [ ]:
normalized_increments = (targets.isel(time=0, drop=True) - inputs.isel(time=0, drop=True)[list(targets.data_vars.keys())]) / diffs_stddev_by_level
normalized_increments

In [ ]:
get_dashboard(normalized_increments, projection=ccrs.Robinson())